# GeoGuessr Country Classifier — Data Demo
This notebook demonstrates how the dataset is loaded, what the inputs look like, and what the targets are.

**Model input:** RGB street view image, resized to 224x224 pixels  
**Model target:** Country label (integer class index mapping to a country name string)

In [ ]:
%pip install torch torchvision kagglehub --quiet

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from collections import Counter, defaultdict
import kagglehub

## 1. Download the dataset
The dataset is downloaded automatically from Kaggle using kagglehub.

In [ ]:
path = kagglehub.dataset_download('ubitquitin/geolocation-geoguessr-images-50k')
DATA_ROOT = os.path.join(path, 'compressed_dataset')
print(f'Dataset path: {DATA_ROOT}')
print(f'Total country folders: {len(os.listdir(DATA_ROOT))}')
print(f'Sample folders: {os.listdir(DATA_ROOT)[:8]}')

## 2. Dataset structure
Images are organized as `compressed_dataset/<country_name>/<image>.jpg`
Each subfolder name becomes a class label automatically via ImageFolder.

In [ ]:
# Show example folder structure
sample_country = os.listdir(DATA_ROOT)[5]
sample_files = os.listdir(os.path.join(DATA_ROOT, sample_country))[:5]
print(f'Example country folder: {sample_country}/')
for f in sample_files:
    print(f'  {f}')

## 3. Class distribution (raw)
The raw dataset has severe class imbalance — the US alone has 12,014 images.

In [ ]:
raw_dataset = datasets.ImageFolder(root=DATA_ROOT, transform=transforms.ToTensor())
label_counts = Counter(raw_dataset.targets)
sorted_counts = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)

print(f'Total images: {len(raw_dataset)}')
print(f'Total countries: {len(raw_dataset.classes)}')
print(f'\nTop 10 countries by image count:')
for idx, count in sorted_counts[:10]:
    print(f'  {raw_dataset.classes[idx]:35s} {count:6d} images')
print(f'\nBottom 10 countries by image count:')
for idx, count in sorted_counts[-10:]:
    print(f'  {raw_dataset.classes[idx]:35s} {count:6d} images')

## 4. Balanced DataLoader
We drop countries with fewer than 50 images and cap countries with more than 1000 images to reduce bias.

In [ ]:
MIN_IMAGES = 50
MAX_IMAGES = 1000

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(root=DATA_ROOT, transform=eval_transform)
label_counts = Counter(full_dataset.targets)
valid_classes = {idx for idx, count in label_counts.items() if count >= MIN_IMAGES}

class_seen = defaultdict(int)
balanced_indices = []
for idx, label in enumerate(full_dataset.targets):
    if label in valid_classes and class_seen[label] < MAX_IMAGES:
        balanced_indices.append(idx)
        class_seen[label] += 1

kept_class_indices = sorted(valid_classes)
old_to_new = {old: new for new, old in enumerate(kept_class_indices)}
classes = [full_dataset.classes[i] for i in kept_class_indices]
num_classes = len(classes)

class RemappedSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, label_map):
        self.dataset = dataset; self.indices = indices; self.label_map = label_map
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        return img, self.label_map[label]

balanced_dataset = RemappedSubset(full_dataset, balanced_indices, old_to_new)
loader = DataLoader(balanced_dataset, batch_size=8, shuffle=True, num_workers=0)

print(f'Balanced dataset: {len(balanced_dataset)} images across {num_classes} countries')
print(f'US images (was 12014): {class_seen[full_dataset.class_to_idx["United States"]]}')

## 5. Visualize a batch
**Input:** Tensor of shape (3, 224, 224) — normalized RGB image  
**Target:** Integer class index (0 to num_classes-1) mapping to a country name

In [ ]:
images, labels = next(iter(loader))

print(f'Input batch shape:  {images.shape}  (batch, channels, height, width)')
print(f'Input dtype:        {images.dtype}')
print(f'Input value range:  [{images.min():.2f}, {images.max():.2f}]  (normalized)')
print(f'Target batch shape: {labels.shape}')
print(f'Target dtype:       {labels.dtype}')
print(f'Target values:      {labels.tolist()}')
print(f'Target labels:      {[classes[l] for l in labels.tolist()]}')

# Visualize
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    img = (images[i] * std + mean).clamp(0, 1)
    ax.imshow(np.transpose(img.numpy(), (1, 2, 0)))
    ax.set_title(f'{classes[labels[i]]}\n(class {labels[i].item()})', fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Inputs and Targets — Street View Images with Country Labels', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Summary
- **Input:** RGB image tensor, shape `(3, 224, 224)`, normalized with ImageNet mean/std
- **Target:** Integer class index in range `[0, 75]` mapping to one of 76 country names
- **Dataset size:** ~38,000 images after balancing (70% train / 15% val / 15% test)
- **Classes:** 76 countries with >= 50 images, capped at 1000 images each